# 🚀 RIVO CLOUD GPU WORKER (Google Colab T4)
### كارت شاشة سحابي مجاني خارق لعزل الأغاني في ثواني لصالح سيرفر Rivo

**طريقة التشغيل:**
1. تأكد من تفعيل كارت الشاشة: من القائمة فوق اضغط **Runtime** -> **Change runtime type** -> اختر **T4 GPU** -> اضغط **Save**.
2. اضغط على زرار التشغيل **▶ (Run)** في الخلية التالية.

In [ ]:
# 1. تثبيت حزمة العزل الذكية لدعم كروت الشاشة
!pip install -q "audio-separator[gpu]" requests tqdm

import os, sys, time, requests, shutil, subprocess
from pathlib import Path

# رابط سيرفر ريفو الرسمي
RIVO_SERVER = "https://rivo-m.duckdns.org"
WORKER_NAME = "Colab-T4-Worker"

print("=" * 65)
print(f"🚀 RIVO CLOUD GPU WORKER [{WORKER_NAME}] STARTED!")
print(f"📡 Connected to Rivo Server: {RIVO_SERVER}")
print("⚡ Model: Kim_Vocal_2 (MDX-Net Architecture)")
print("🎧 Waiting for isolation jobs from Rivo queue...")
print("=" * 65)

os.makedirs("worker_temp", exist_ok=True)
os.makedirs("models", exist_ok=True)

consecutive_fails = 0

while True:
    try:
        # فحص إذا كان هناك أغنية تحتاج لعزل في السيرفر
        res = requests.get(f"{RIVO_SERVER}/api/tracks/vocal-worker/next?worker={WORKER_NAME}", timeout=12)
        if res.status_code == 200:
            data = res.json()
            if data.get("hasJob"):
                consecutive_fails = 0
                stem_key = data["stemKey"]
                audio_url = data["audioUrl"]
                title = data.get("title", stem_key)
                aliases = data.get("aliases", [])

                print(f"\n🎵 [New Job Received] Isolating: {title}")
                print(f"   Stem Key: {stem_key}")
                t0 = time.time()

                # 1. تحميل ملف الصوت الأصلي
                input_file = f"worker_temp/{stem_key}_input.mp3"
                print("⬇️  Downloading source audio...")
                with requests.get(audio_url, stream=True, timeout=90) as r:
                    r.raise_for_status()
                    with open(input_file, 'wb') as f:
                        shutil.copyfileobj(r.raw, f)

                # 2. العزل الفائق باستخدام كارت الشاشة وموديل Kim_Vocal_2
                print("⚡ Isolating pure vocals on NVIDIA T4 GPU (Kim_Vocal_2)... [takes ~12s]")
                out_dir = f"worker_temp/{stem_key}_out"
                os.makedirs(out_dir, exist_ok=True)

                cmd = [
                    "audio-separator",
                    input_file,
                    "--model_filename", "Kim_Vocal_2.onnx",
                    "--single_stem", "Vocals",
                    "--mdx_overlap", "0.25",
                    "--output_dir", out_dir,
                    "--output_format", "MP3",
                    "--model_file_dir", "models"
                ]
                subprocess.run(cmd, check=True)

                # 3. إيجاد ملف الـ Vocals المنفصل
                vocal_file = None
                for fname in os.listdir(out_dir):
                    if "vocals" in fname.lower() and fname.endswith(".mp3"):
                        vocal_file = os.path.join(out_dir, fname)
                        break

                if vocal_file and os.path.exists(vocal_file):
                    dur = time.time() - t0
                    print(f"✅ Finished in {dur:.1f}s! Uploading result back to Rivo...")
                    with open(vocal_file, 'rb') as vf:
                        upload_res = requests.post(
                            f"{RIVO_SERVER}/api/tracks/vocal-worker/complete",
                            files={"vocalFile": vf},
                            data={"stemKey": stem_key, "aliases": ",".join(aliases), "worker": WORKER_NAME},
                            timeout=90
                        )
                    print(f"🎉 SUCCESS! Vocal stem stored permanently on server! (HTTP {upload_res.status_code})")
                    shutil.rmtree(out_dir, ignore_errors=True)
                    if os.path.exists(input_file):
                        os.remove(input_file)
                else:
                    print("❌ Error: Isolated vocal file not found in output directory.")
            else:
                time.sleep(3)
        else:
            time.sleep(5)
    except requests.exceptions.RequestException as re:
        time.sleep(5)
    except Exception as e:
        print(f"⚠️  Worker error: {e}")
        time.sleep(5)
